# Xpect AI — RAG + LLM Generation

This notebook builds Phase 2 of the Xpect AI RAG system.

## Phase 2 Pipeline

User Question
→ Query Embedding
→ FAISS Retrieval
→ Relevant Movie Context
→ Prompt Construction
→ LLM
→ Natural Language Answer

Phase 1 built the retrieval foundation.

Phase 2 connects the retrieved information to an LLM so that
Xpect AI can generate natural-language answers.

##### Artifacts

In [16]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
print("Done")

c:\Users\DELL\.venvs\xpect-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Done


##### Loading the existing vector store

In [17]:
index=faiss.read_index(
    "../vector_store/netflix_index"
)
with open("../vector_store/documents.pkl","rb") as f:
    documents=pickle.load(f)
movies = pd.read_pickle(
    "../vector_store/movies.pkl"
)

print("Vector store loaded successfully!")
print("Movies:", len(documents))
print("Vectors:", index.ntotal)

Vector store loaded successfully!
Movies: 8807
Vectors: 8807


##### Loading Embedding Modek

In [18]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
print("Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 711.97it/s]


Embedding model loaded!


##### Making Retriver

In [19]:
def retrieve_movies(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

#### Test the query

In [20]:
results = retrieve_movies(
    "movies about a father and daughter"
)
for movie in results:
    print(movie)
    print("-" * 80)


Title : Father of the Year
Type : Movie
Director : Unknown
Cast : David Spade, Nat Faxon, Joey Bragg, Matt Shively, Bridgit Mendler, Jackie Sandler, Mary Gillis
Country : United States
Release Year : 2018
Rating : TV-MA
Duration : 95 min
Genres : Comedies
Description : A drunken debate between two recent college grads about whose father would win in a fight leads to mayhem when their dads take the challenge seriously.

--------------------------------------------------------------------------------

Title : Fatherhood
Type : Movie
Director : Paul Weitz
Cast : Kevin Hart, Alfre Woodard, Lil Rel Howery, DeWanda Wise, Frankie Faison, Anthony Carrigan, Paul Reiser, Melody Hurd
Country : United States
Release Year : 2021
Rating : PG-13
Duration : 111 min
Genres : Dramas
Description : A widowed new dad copes with doubts, fears, heartache and dirty diapers as he sets out to raise his daughter on his own. Inspired by a true story.

-------------------------------------------------------------

### Context Layer

In [22]:
context="\n\n -- \n\n".join(results)
print(context)


Title : Father of the Year
Type : Movie
Director : Unknown
Cast : David Spade, Nat Faxon, Joey Bragg, Matt Shively, Bridgit Mendler, Jackie Sandler, Mary Gillis
Country : United States
Release Year : 2018
Rating : TV-MA
Duration : 95 min
Genres : Comedies
Description : A drunken debate between two recent college grads about whose father would win in a fight leads to mayhem when their dads take the challenge seriously.


 -- 


Title : Fatherhood
Type : Movie
Director : Paul Weitz
Cast : Kevin Hart, Alfre Woodard, Lil Rel Howery, DeWanda Wise, Frankie Faison, Anthony Carrigan, Paul Reiser, Melody Hurd
Country : United States
Release Year : 2021
Rating : PG-13
Duration : 111 min
Genres : Dramas
Description : A widowed new dad copes with doubts, fears, heartache and dirty diapers as he sets out to raise his daughter on his own. Inspired by a true story.


 -- 


Title : Dear Dad
Type : Movie
Director : Tanuj Bhramar
Cast : Arvind Swamy, Himanshu Sharma, Ekavali Khanna, Aman Uppal, Bhavik

### RAG orchestration

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
from src.llm import generate_answer
print("done")

done


In [26]:
def ask_xpect(query):
    result=retrieve_movies(query)
    context="\n\n -- \n\n".join(result)
    answer=generate_answer(query,context)
    return answer
answer = ask_xpect("movies about a father and daughter")
print(answer)

**Fatherhood (2021)**  
- **Director:** Paul Weitz  
- **Cast:** Kevin Hart, Alfre Woodard, Lil Rel Howery, DeWanda Wise, Frankie Faison, Anthony Carrigan, Paul Reiser, Melody Hurd  
- **Country:** United States  
- **Rating:** PG‑13  
- **Duration:** 111 min  
- **Genres:** Drama  
- **Description:** A widowed new dad copes with doubts, fears, heartache and dirty diapers as he sets out to raise his daughter on his own. Inspired by a true story.

This is the only film in the provided context that centers on a father‑daughter relationship.
